# 01 — Data Profiling (Phase 1, Task 4)

**เป้าหมาย:** ยืนยันว่าปัญหาข้อมูล 17 ข้อที่บันทึกไว้ใน `CLAUDE.md` §6 เป็นเรื่องจริง ตรงกับตัวเลขที่บันทึกไว้หรือไม่
โดย **สังเกตการณ์เท่านั้น (observe-only)** — Phase 1 ยังไม่แก้ไขข้อมูลใด ๆ ทั้งสิ้น

**กติกาของ notebook นี้ (ตาม CLAUDE.md N2):** logic ทั้งหมดอยู่ใน `src/rri/utils/profiling.py`
notebook นี้ทำหน้าที่แค่ import → เรียกใช้ → แสดงผล → อธิบาย ไม่มี business logic อยู่ใน cell ใดเลย

**โครงสร้าง:**
1. Metadata ระดับไฟล์ (ขนาด, encoding, delimiter, line ending) ของทั้ง 9 ไฟล์
2. Profile รายคอลัมน์ของไฟล์อุบัติเหตุ (ไฟล์หลัก 161,307 แถว)
3. ข้อมูลซ้ำ (duplicates) — ยืนยัน P5
4. คอลัมน์แฝด (twin columns / null-correlated pairs) — ยืนยัน P3
5. รูปแบบวันที่/เวลาที่ปะปนกัน — ยืนยัน P1, P2
6. จำนวนหมวดหมู่ (category cardinality) — ยืนยัน P4
7. Profile ไฟล์ CSV ที่เหลือ
8. Profile โครงสร้างไฟล์ JSON ทั้ง 3 ไฟล์
9. สรุปผล — รายงานที่ตรงกับ CLAUDE.md ทั้งหมด และสิ่งที่พบใหม่ที่ยังไม่เคยบันทึกไว้

รายงานฉบับสมบูรณ์ (ทุกคอลัมน์ของทุกไฟล์) อยู่ที่ `reports/data_profiling_report.html`
(สร้างโดย `scripts/build_data_profiling_report.py`) — notebook นี้เน้นเล่าเรื่องสิ่งที่สำคัญที่สุด

## 0. Setup

Import ฟังก์ชัน profiling จาก `src/` และตั้งค่า logging (ตาม CLAUDE.md N8: ห้ามใช้ `print()`)

In [1]:
import sys
from pathlib import Path

# Local dev (VS Code): notebook runs from notebooks/, so the repo root needs
# to be on sys.path for `config` and `rri` to import. In Colab this becomes
# the Drive-mounted project path instead (see config/settings.py).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl

from config.settings import settings
from rri.utils.logging import configure_logging
from rri.utils.profiling import (
    check_duplicate_date_string_consistency,
    check_value_when_both_populated,
    check_within_group_agreement,
    compute_dedup_impact,
    count_resolved_by_normalization,
    discriminate_transposed_date_pairs,
    find_conflicting_groups,
    find_transposed_date_string_pairs,
    load_json_file,
    parse_slash_date,
    population_rate_by_group,
    profile_csv_file,
    profile_date_column_formats,
    profile_file_metadata,
    profile_json_structure,
    profile_slash_date_side_evidence,
    profile_time_column_formats,
    sample_deterministic,
)

configure_logging()
settings.raw_dir

PosixPath('/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/00_raw')

## 1. Metadata ระดับไฟล์ (ทั้ง 9 ไฟล์)

ตรวจ ขนาดไฟล์, encoding, BOM (`﻿`), delimiter, และรูปแบบ line ending ของทุกไฟล์ในคราวเดียว
ผลที่คาดว่าจะเจอตาม CLAUDE.md P7: **ทุกไฟล์ CSV มี BOM** (`encoding='utf-8-sig'` จำเป็นเสมอ)

In [2]:
ALL_FILENAMES = [
    "thailand_accidents_2019_2026_combined.csv",
    "thailand_rainfall_2021_2026_combined.csv",
    "thailand_flood_risk_areas.csv",
    "thailand_heavy_rain_R35mm.csv",
    "stt_car_fuel_at_25690228.csv",
    "car_registered_new_2565_2569_combined.csv",
    "public_holidays_all.json",
    "thai_bank_holidays_all.json",
    "tat_tourist_attractions.json",
]

metadata_rows = [
    {
        "filename": m.filename,
        "size_mb": round(m.size_bytes / 1_000_000, 2),
        "encoding": m.encoding,
        "has_bom": m.has_bom,
        "delimiter": m.delimiter,
        "line_ending": m.line_ending,
    }
    for m in (profile_file_metadata(settings.raw_dir / f) for f in ALL_FILENAMES)
]
pl.DataFrame(metadata_rows)

filename,size_mb,encoding,has_bom,delimiter,line_ending
str,f64,str,bool,str,str
"""thailand_accidents_2019_2026_c…",85.58,"""utf-8-sig""",true,""",""","""CRLF"""
"""thailand_rainfall_2021_2026_co…",1.69,"""utf-8-sig""",true,""",""","""CRLF"""
"""thailand_flood_risk_areas.csv""",6.12,"""utf-8-sig""",true,""",""","""CRLF"""
"""thailand_heavy_rain_R35mm.csv""",0.08,"""utf-8-sig""",true,""",""","""CRLF"""
"""stt_car_fuel_at_25690228.csv""",0.7,"""utf-8""",false,""",""","""CRLF"""
"""car_registered_new_2565_2569_c…",0.12,"""utf-8-sig""",true,""",""","""CRLF"""
"""public_holidays_all.json""",0.02,"""utf-8""",false,null,"""LF"""
"""thai_bank_holidays_all.json""",0.01,"""utf-8""",false,null,"""LF"""
"""tat_tourist_attractions.json""",7.32,"""utf-8""",false,null,"""LF"""


## 2. Profile รายคอลัมน์ — ไฟล์อุบัติเหตุ (ไฟล์หลัก)

`profile_csv_file` อ่านทุกคอลัมน์เป็น string ล้วน (ไม่เดา dtype) แล้วคำนวณ %null, จำนวนค่าที่ไม่ซ้ำ,
ตัวอย่างค่า, และ min/max (เฉพาะคอลัมน์ตัวเลข) — ใช้ candidate key `(ACC_CODE, ปีที่เกิดเหตุ)` ตาม P5
(`ACC_CODE` อย่างเดียวไม่ใช่ key จริง เพราะถูกใช้ซ้ำข้ามปี)

**ผลที่คาดว่าจะได้:** 161,307 แถว × 45 คอลัมน์ ตรงตาม CLAUDE.md §5

In [3]:
accidents_profile = profile_csv_file(
    settings.raw_dir / "thailand_accidents_2019_2026_combined.csv",
    candidate_key_columns=["ACC_CODE", "ปีที่เกิดเหตุ"],
)

# N8: no print() -- structlog only. A one-row DataFrame displays the shape
# via the notebook's own cell output instead of stdout.
display(  # noqa: F821 -- `display` is a Jupyter builtin, available at runtime
    pl.DataFrame([{"rows": accidents_profile.row_count, "columns": accidents_profile.column_count}])
)

pl.DataFrame(
    [
        {
            "column": c.name,
            "dtype": c.dtype,
            "null_pct": round(c.null_pct, 1),
            "distinct": c.distinct_count,
            "sample": ", ".join(c.sample_values[:3]),
        }
        for c in accidents_profile.columns
    ]
)

rows,columns
i64,i64
161307,45


column,dtype,null_pct,distinct,sample
str,str,f64,i64,str
"""ปีจากชื่อไฟล์""","""String""",0.0,8,"""2019, 2019, 2019"""
"""source_file""","""String""",0.0,8,"""accident2019.csv, accident2019…"
"""ปีที่เกิดเหตุ""","""String""",0.0,8,"""2019, 2019, 2019"""
"""วันที่เกิดเหตุ""","""String""",0.0,2678,"""1/1/2019, 1/1/2019, 1/1/2019"""
"""เวลา""","""String""",0.0,2853,"""0:00, 0:03, 0:05"""
…,…,…,…,…
"""บริเวณที่เกิดเหตุอื่น""","""String""",94.8,60,"""ทางตรง+ที่ราบ, ทางยกระดับ+ที่ร…"
"""รถยนต์นั่งส่วนบุคคล/รถยนต์นั่ง…","""String""",94.6,9,"""0, 1, 0"""
"""รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้…","""String""",94.6,3,"""0, 0, 0"""


## 3. ข้อมูลซ้ำ — ยืนยัน P5

CLAUDE.md บันทึกไว้ว่า: แถวซ้ำแบบเป๊ะ 82 แถว, `ACC_CODE` ซ้ำ 1,702 ครั้ง (ข้ามปี ใช้ซ้ำได้ จึง key จริงคือ
`(ACC_CODE, year)`), และ `ACC_CODE` เป็น null 17.3% ของแถวทั้งหมด (จึงเป็น primary key ไม่ได้)

In [4]:
dup = accidents_profile.duplicates
excess_rows = dup.candidate_key_duplicate_row_count - dup.candidate_key_duplicate_group_count

pl.DataFrame(
    [
        {
            "full_row_duplicates": dup.full_row_duplicate_count,
            "candidate_key": " + ".join(dup.candidate_key_columns or ()),
            "duplicate_groups": dup.candidate_key_duplicate_group_count,
            "excess_rows": excess_rows,
            "null_key_rows": dup.candidate_key_null_count,
        }
    ]
)

full_row_duplicates,candidate_key,duplicate_groups,excess_rows,null_key_rows
i64,str,i64,i64,i64
82,"""ACC_CODE + ปีที่เกิดเหตุ""",1641,1702,27932


### พบสิ่งที่ต่างจาก CLAUDE.md P5 — ต้องรายงาน

`full_row_duplicates = 82` และ `excess_rows = 1,702` (แถวส่วนเกินหลังหักแถวแรกของแต่ละกลุ่ม
`3,343 − 1,641 = 1,702`) **ตรงกับตัวเลขที่บันทึกไว้ทุกตัว** ✅

แต่สมมติฐานที่ CLAUDE.md ใช้อธิบายว่า *"ทำไมต้องใช้ `(ACC_CODE, year)` เป็น key จริง"* — คือ
*"ACC_CODE ถูกใช้ซ้ำข้ามปีที่ต่างกัน"* — **ตรวจสอบแล้วไม่จริง**: ไม่มี `ACC_CODE` ตัวไหนเลยที่ปรากฏ
มากกว่า 1 ปี ดังนั้น **key ผสม `(ACC_CODE, year)` ไม่ได้แก้ปัญหาความซ้ำเลย** — จำนวนกลุ่มซ้ำเท่ากันทั้งใช้
`ACC_CODE` เดี่ยว ๆ และใช้คู่กับปี (1,641 กลุ่มทั้งคู่)

การพบนี้เพียงอย่างเดียวยังไม่พอ — มันบอกแค่ว่าคำอธิบายเดิมผิด ไม่ได้บอกว่าอะไรถูก ส่วนถัดไปตรวจสอบ
ต่อให้ลึกขึ้น: หน่วยงานเดียวกันหรือคนละหน่วยงาน? ตัวเลขผู้บาดเจ็บ/เสียชีวิตตรงกันหรือขัดแย้งกัน?
ยอดผู้เสียชีวิตหลัง dedup เป็นเท่าไหร่?

### 3.1 การสืบสวนเพิ่มเติม — หน่วยงาน, ความเห็นตรงกันของยอดผู้บาดเจ็บ, ผลกระทบต่อยอด reconciliation

ตรวจ 3 อย่างกับทุก 1,641 กลุ่มที่ซ้ำ (ตรวจครบทุกกลุ่ม ไม่ใช่สุ่มตัวอย่าง):
1. `หน่วยงาน` — ตรงกันในทุกแถวของกลุ่มไหม (ถ้าไม่ตรง = คนละหน่วยงานรายงานซ้ำ = ปัญหาเชิงระบบ ไม่ใช่ data entry)
2. `ผู้เสียชีวิต` — ตัวเลขตรงกันในทุกแถวของกลุ่มไหม (ถ้าตรงกันเสมอ แปลว่ายอด reconciliation ไม่มีความกำกวม)
3. ผลของการ dedup (เหลือ 1 แถวต่อกลุ่ม) ต่อยอดผู้เสียชีวิตรวม — คำนวณทั้งก่อนและหลัง ไม่เลือกว่าอันไหนถูก

In [5]:
accidents_raw = pl.read_csv(
    settings.raw_dir / "thailand_accidents_2019_2026_combined.csv",
    encoding="utf8-lossy",
    infer_schema_length=0,
    null_values=["", "NA", "N/A", "-"],
)

agency_agreement = check_within_group_agreement(accidents_raw, ["ACC_CODE"], "หน่วยงาน")
fatality_agreement = check_within_group_agreement(accidents_raw, ["ACC_CODE"], "ผู้เสียชีวิต")

pl.DataFrame(
    [
        {
            "check": "agency (หน่วยงาน)",
            "groups_checked": agency_agreement.groups_checked,
            "agree": agency_agreement.agree_count,
            "conflict": agency_agreement.conflict_count,
        },
        {
            "check": "fatalities (ผู้เสียชีวิต)",
            "groups_checked": fatality_agreement.groups_checked,
            "agree": fatality_agreement.agree_count,
            "conflict": fatality_agreement.conflict_count,
        },
    ]
)

check,groups_checked,agree,conflict
str,i64,i64,i64
"""agency (หน่วยงาน)""",1641,1641,0
"""fatalities (ผู้เสียชีวิต)""",1641,1641,0


In [6]:
fatality_impact = compute_dedup_impact(accidents_raw, "ACC_CODE", ["ผู้เสียชีวิต"])

pl.DataFrame(
    [
        {
            "rows_before": fatality_impact.rows_before,
            "rows_after": fatality_impact.rows_after,
            "fatalities_before": fatality_impact.sums_before["ผู้เสียชีวิต"],
            "fatalities_after": fatality_impact.sums_after["ผู้เสียชีวิต"],
        }
    ]
)

rows_before,rows_after,fatalities_before,fatalities_after
i64,i64,f64,f64
161307,159605,21814.0,21497.0


### สรุป P5 — ข้อเท็จจริงครบ ยังไม่ได้เลือกกฎ dedup (รอเจ้าของโปรเจกต์ตัดสินใจ)

- **หน่วยงานตรงกัน 100%** (1,641/1,641) — ตัดความเป็นไปได้ที่สงสัยไว้ (สองหน่วยงานรายงานอุบัติเหตุ
  เดียวกันที่ทางหลวงตัดกับทางชนบท) ออกไปได้เลย ทุกคู่ซ้ำเป็นของหน่วยงานเดียวกันเอง
- **ยอดผู้เสียชีวิตตรงกัน 100%** (1,641/1,641) — **ไม่มีความขัดแย้งเลย** จังหวัด/ถนน/กม./ปี ก็ตรงกัน
  100% เช่นกัน (ตรวจแล้วแยกต่างหาก) แปลว่ายอด reconciliation ที่เป็นตัวเลข **ไม่กำกวม** ไม่ว่าจะเลือก
  แถวไหนในกลุ่มไว้ก็ได้ยอดเดียวกัน — ความกำกวมอยู่ที่คอลัมน์บรรยายข้อความเท่านั้น

**ยอด reconciliation — ใช้ 2 gate แยกกัน ไม่ใช่เลือกอันเดียว (บันทึกไว้ใน CLAUDE.md §8):**
- **Gate 1 — หลัง Extract (ก่อน dedup) = 21,814** พิสูจน์ว่าไม่มีอะไรหายไปตอน ingest ข้อมูลดิบ
- **Gate 2 — หลัง Load (หลัง dedup) = 21,497** (ต่างกัน 317, แม่นยำเพราะไม่มีความขัดแย้งของตัวเลขใน
  แต่ละกลุ่มเลย) พิสูจน์ว่า dedup ตัดออกไปตามที่ตั้งใจพอดี ไม่มากไม่น้อยกว่านั้น

ทั้งสอง gate ตรวจคนละอย่าง ยุบเหลือตัวเดียวจะเสียการตรวจสอบไปครึ่งหนึ่ง

สาเหตุที่วันที่ในบางกลุ่มไม่ตรงกัน (448 จาก 1,641 กลุ่ม) ก็ตรวจสอบแล้วเช่นกัน (ใช้ format ที่ถูกต้องตาม
P1 ต่อไฟล์ ไม่ใช่เดาแบบลองทั้งสองฟอร์แมต) — **ทั้ง 448 กลุ่มเป็นการสลับตำแหน่งวัน/เดือนแบบเป๊ะ**
(เช่น `4/12/2023` กับ `12/4/2023` ในไฟล์เดียวกันที่เป็น MDY = เมษายน 12 กับ ธันวาคม 4) เป็นรูปแบบ
ความผิดพลาดที่ระบุได้ชัดเจน ไม่ใช่ความเสียหายแบบสุ่ม — สนับสนุนว่านี่คือเหตุการณ์เดียวกันที่บันทึกซ้ำจริง
ประเด็นที่สำคัญกว่านั้น (ทั้ง 448 คู่นี้อยู่ไฟล์เดียวกันหรือคนละไฟล์ — กระทบความถูกต้องของสมมติฐานทั้ง
P1) ตรวจต่อในหัวข้อถัดไป

### 3.2 คำถามสำคัญที่สุด — สมมติฐาน "หนึ่งไฟล์ หนึ่งรูปแบบวันที่" (P1) ยังใช้ได้ไหม?

ทั้ง pipeline การแก้ P1 (45,417 แถวที่กำกวม) วางอยู่บนสมมติฐานเดียว: **ตรวจรูปแบบต่อไฟล์ ไม่ใช่ต่อแถว**
ถ้า 448 คู่ที่วันที่สลับกันอยู่ **คนละไฟล์กัน** สมมติฐานนี้ยิ่งแข็งแรงขึ้น (แต่ละไฟล์ยังคงมีรูปแบบเดียว
ที่สอดคล้องในตัวเอง) แต่ถ้าอยู่ **ไฟล์เดียวกัน** ต้องพิจารณาว่าไฟล์นั้นแอบผสมสองรูปแบบจริงหรือไม่

In [7]:
date_conflicts = find_conflicting_groups(accidents_raw, ["ACC_CODE"], "วันที่เกิดเหตุ")
conflicting_codes = [key[0] for key, _ in date_conflicts]

subset = accidents_raw.filter(pl.col("ACC_CODE").is_in(conflicting_codes))
source_check = check_within_group_agreement(subset, ["ACC_CODE"], "source_file")

by_file: dict[str, int] = {}
for code in conflicting_codes:
    f = accidents_raw.filter(pl.col("ACC_CODE") == code)["source_file"][0]
    by_file[f] = by_file.get(f, 0) + 1

pl.DataFrame(
    [
        {
            "date_conflicting_groups": len(date_conflicts),
            "same_source_file": source_check.agree_count,
            "different_source_file": source_check.conflict_count,
            "by_file": str(by_file),
        }
    ]
)

date_conflicting_groups,same_source_file,different_source_file,by_file
i64,i64,i64,str
448,448,0,"""{'accident2023.csv': 448}"""


### คำตอบ: เดียวกันทั้งหมด — และทั้งหมดกระจุกอยู่ที่ `accident2023.csv` ไฟล์เดียว — **ยืนยันแล้ว สมมติฐานยังใช้ได้**

**ทั้ง 448 คู่อยู่ในไฟล์เดียวกันเสมอ (100%) และทั้งหมดอยู่ใน `accident2023.csv` โดยเฉพาะ** — ไม่มีคู่ไหน
กระจายไปไฟล์อื่นเลย ซึ่งเป็นกรณีที่ปกติแล้วควรตีความว่า "สมมติฐานพัง ต้อง redesign"

**เหตุผลที่ทำให้สมมติฐานยังใช้ได้ — ยืนยันแล้ว ไม่ใช่แค่หลักฐานเฉย ๆ:** ถ้า `accident2023.csv` ผสมสองรูปแบบ
จริง ๆ การผสมนั้นต้องกระจายอยู่ทั้งไฟล์ ไม่ใช่กระจุกอยู่แค่ในคู่ duplicate — และเราจะเห็นบางแถวที่มีตัวเลข
ด้านแรก > 12 (หลักฐานของ DMY) กระจายอยู่ทั้งไฟล์ด้วย ต่อไปตรวจโดยตรง (ดูเซลล์ถัดไป) พบว่า **ไม่มีเลย**

ส่วน 448 คู่เองก็ไม่ได้เป็นหลักฐานคัดค้าน: ทุกคู่มีทั้งสองด้าน ≤ 12 เสมอ (ทั้งสองสตริง) — ตามกลไกของมันเอง
ถ้าสลับตำแหน่งแล้วยังอ่านเป็นวันที่ที่ดูสมเหตุสมผลได้ทั้งคู่ ทั้งสองด้านต้อง ≤ 12 อยู่แล้ว แปลว่า
**ข้อผิดพลาดกลุ่มนี้ไม่มีทางเป็นหลักฐานคัดค้านรูปแบบของไฟล์ได้เลย** (เกิดเฉพาะในช่วงที่กำกวมอยู่แล้ว) และ
ไม่มีทางปนเปื้อนคะแนนเสียงข้างมาก (ซึ่งมาจากแถวไม่กำกวมล้วน ๆ)

รูปแบบของการสลับ (`a/b/y` ↔ `b/a/y` แบบเป๊ะ 100%) ตรงกับ**ข้อผิดพลาดตอนคีย์ข้อมูลซ้ำ** (พิมพ์วันที่เดียวกัน
สองครั้ง สลับวัน/เดือนโดยไม่ตั้งใจในครั้งที่สอง) มากกว่าไฟล์ผสมรูปแบบจริง

In [8]:
acc2023_dates = (
    accidents_raw.filter(pl.col("source_file") == "accident2023.csv")["วันที่เกิดเหตุ"]
    .drop_nulls()
    .to_list()
)

pl.DataFrame([profile_slash_date_side_evidence(acc2023_dates)])

ambiguous,mdy_evidence
i64,i64
9263,14578


**ผลตรง**: `mdy_evidence` (ตัวเลขด้านที่สอง > 12) = 14,578 แถว ตรงกับตาราง P1 เป๊ะ ส่วน `dmy_evidence`
(ด้านแรก > 12) = **0 แถว** — ไม่ใช่แค่คะแนนเสียงข้างมาก แต่เป็น**ศูนย์หลักฐานคัดค้านทั้งไฟล์** สรุปได้ว่า
`accident2023.csv` เป็น MDY ล้วน ไม่มีข้อยกเว้น — สมมติฐาน per-file **ยืนยันแล้ว ไม่ต้อง redesign**

---

### แก้ไข safety net ข้อที่ 4 ที่เสนอไว้ก่อนหน้านี้ — เดิมเขียนผิด

ร่างเดิมเขียนว่า *"ทั้งสองแถวในคู่ duplicate ที่รู้กันต้อง resolve เป็นวันที่เดียวกันหลัง parse จริง"* — **ผิด**
เพราะเมื่อยืนยันแล้วว่าไฟล์เป็น MDY ล้วน (ด้านบน) ทั้งสองแถวของคู่ที่สลับตำแหน่งกันจะ parse เป็น MDY เสมอ
แถวหนึ่งได้ `month=a day=b` อีกแถวได้ `month=b day=a` — **รับประกันทางคณิตศาสตร์ว่าต้องได้วันที่ต่างกัน**
กฎที่บังคับให้ตรงกันจะ flag คู่ทั้ง 448 คู่ว่า parser พังทุกครั้งไป และถ้ามีใครเชื่อสัญญาณนั้นแล้วไป "แก้"
parser ที่จริง ๆ ทำงานถูกอยู่แล้ว จะยิ่งพังกว่าเดิม

ปัญหาจริงของ 448 คู่นี้ไม่ใช่เรื่องรูปแบบวันที่ — แต่เป็นการคีย์ข้อมูลซ้ำแล้วสลับวัน/เดือนโดยไม่ตั้งใจ
ต้องแยกเป็น 2 แนวคิดที่ไม่ทับกัน:

- **Safety net 4a (การตรวจ parser จริง):** เฉพาะกลุ่ม duplicate ที่ raw string **เหมือนกันเป๊ะ** ทุกแถว —
  parser ที่ deterministic ต้องได้วันที่เดียวกันเสมอ ถ้าไม่ตรงคือบั๊กจริง
- **รายงาน `date_ambiguous_pairs` (ไม่ใช่การตรวจ parser):** กลุ่มที่ raw string สลับตำแหน่งกันเป๊ะ (448 คู่)
  เป็นปัญหาการคีย์ข้อมูล ต้องแยกออกไปอีกรายงานหนึ่ง แก้ด้วยวิธีอื่น (ดูด้านล่าง)

In [9]:
acc2023_parsed = accidents_raw.filter(pl.col("source_file") == "accident2023.csv").with_columns(
    pl.col("วันที่เกิดเหตุ")
    .map_elements(lambda v: parse_slash_date(v, month_first=True), return_dtype=pl.Date)
    .alias("parsed_date")
)

safety_net_4a = check_duplicate_date_string_consistency(
    acc2023_parsed, "ACC_CODE", "วันที่เกิดเหตุ", "parsed_date"
)

pl.DataFrame(
    [
        {
            "groups_checked (identical raw string)": safety_net_4a.groups_checked,
            "mismatched_groups": safety_net_4a.mismatched_groups,
        }
    ]
)

groups_checked (identical raw string),mismatched_groups
i64,i64
1193,0


**Safety net 4a ผ่านสะอาด**: 1,193 กลุ่ม (duplicate ที่ raw string เหมือนกันเป๊ะ) ทุกกลุ่ม parse
ได้วันที่เดียวกัน ไม่มี mismatch เลย — ยืนยันว่า parser (สมมติฐาน MDY ต่อไฟล์) ทำงานถูกต้อง สอดคล้องกับ
1,193 + 448 = 1,641 ครบทุกกลุ่ม duplicate ของ `ACC_CODE`

### แก้ 448 คู่ — ใช้ safety net #2 (accident_date ≤ report_date, gap ≤ 90 วัน) เป็นตัวชี้ขาด

สำหรับแต่ละคู่ ตัวเลือกวันที่ทั้งสอง (จาก raw string สองแบบที่สลับกัน) ถูกตรวจกับกฎนี้โดยใช้ `วันที่รายงาน`
ของแถวตัวเอง — ตัวเลือกจะ valid ก็ต่อเมื่อ**ทุกแถว**ที่มี raw string นั้นผ่านเงื่อนไข (ครอบคลุมกรณี 23 กลุ่ม
ที่มี 3 แถว ไม่ใช่แค่กรณี 2 แถวง่าย ๆ)

In [10]:
transposed_keys = find_transposed_date_string_pairs(
    accidents_raw.filter(pl.col("ACC_CODE").is_not_null()), "ACC_CODE", "วันที่เกิดเหตุ"
)

discrimination = discriminate_transposed_date_pairs(
    accidents_raw.filter(pl.col("ACC_CODE").is_not_null()),
    "ACC_CODE",
    transposed_keys,
    "วันที่เกิดเหตุ",
    "วันที่รายงาน",
    month_first=True,
)

pl.DataFrame(
    [
        {
            "transposed_pairs_found": len(transposed_keys),
            "resolved_unique": discrimination.resolved_unique,
            "both_valid (undecidable)": discrimination.both_valid,
            "neither_valid (flag)": discrimination.neither_valid,
            "sum": discrimination.resolved_unique
            + discrimination.both_valid
            + discrimination.neither_valid,
        }
    ]
)

transposed_pairs_found,resolved_unique,both_valid (undecidable),neither_valid (flag),sum
i64,i64,i64,i64,i64
448,222,196,30,448


**ผล**: 448 คู่ตรงกับที่พบใน P5 เป๊ะ (425 กลุ่ม 2 แถว + 23 กลุ่ม 3 แถว)

| ผลลัพธ์ | จำนวน | ความหมาย |
|---|---|---|
| Resolved uniquely | 222 | มีตัวเลือกเดียวผ่านเงื่อนไข → นั่นคือวันที่จริง อีกแถวคือข้อผิดพลาดตอนคีย์ |
| Both valid | 196 | ตัดสินไม่ได้จากหลักฐานนี้ — ต้อง flag ไม่เดา |
| Neither valid | 30 | มีอย่างอื่นผิดด้วย (ตรวจตัวอย่างแล้วพบว่าวันที่รายงานเองก็ห่างผิดปกติจากทั้งสองตัวเลือก
เช่น `ACC_CODE 7736329`: ตัวเลือก 2 ม.ค. / 1 ก.พ. แต่รายงานวันที่ 27 ก.ค. — ห่างเกิน 90 วันทั้งคู่) |

### ข้อเสนอสำหรับ Phase 2 (ยังไม่ implement — เป็นแค่ design note)

สำหรับ 196 + 30 = 226 กลุ่ม (452 แถว) ที่ safety net #2 ตัดสินไม่ได้ ให้เพิ่มคอลัมน์ `date_certainty`
(`certain` / `ambiguous`) ใน `fact_accident` แทนการเลือกวันที่แบบเดาสุ่ม — 448 จาก 23,841 แถวใน
`accident2023.csv` คิดเป็น ~1.9% ถือว่าน้อย แต่วันที่ที่ผิดไปเป็นเดือนคือความเสียหายที่กระทบ BQ3
(ความเสี่ยงช่วงวันหยุด) โดยตรง จึงต้อง flag ไว้ ไม่ใช่เดาแล้วปล่อยผ่าน

## 4. คอลัมน์แฝด (twin columns) — ยืนยัน P3

`find_null_correlated_column_groups` หาคู่คอลัมน์ที่ **ไม่เคยมีค่าพร้อมกันในแถวเดียวกันเลย** และรวมกันแล้ว
ครอบคลุมข้อมูลส่วนใหญ่ — สัญญาณของ "ชื่อคอลัมน์เก่า vs ชื่อคอลัมน์ใหม่" ตาม P3
(รายงานเป็น**คู่**ไม่ใช่กลุ่มรวม เพราะคอลัมน์ประเภทยานพาหนะหลายคอลัมน์ก็ไม่เคยมีค่าพร้อมกันเช่นกัน
(field เดียวแต่กระจายเป็นหลายคอลัมน์แบบ one-hot) — ถ้ารวมเป็นกลุ่มเดียวจะกลืนสัญญาณที่ต้องการ)

In [11]:
pl.DataFrame([{"column_a": a, "column_b": b} for a, b in accidents_profile.null_correlated_pairs])

column_a,column_b
str,str
"""ACC_CODE""","""บริเวณที่เกิดเหตุอื่น"""
"""ACC_CODE""","""รถยนต์นั่งส่วนบุคคล/รถยนต์นั่ง…"
"""ACC_CODE""","""รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้…"
"""ACC_CODE""","""รถบรรทุกมากกว่า10ล้อ(รถพ่วง)"""
"""ACC_CODE""","""อื่นๆ"""
…,…
"""รถอื่นๆ""","""บริเวณที่เกิดเหตุอื่น"""
"""รถอื่นๆ""","""รถยนต์นั่งส่วนบุคคล/รถยนต์นั่ง…"
"""รถอื่นๆ""","""รถบรรทุกมากกว่า6ล้อไม่เกิน10ล้…"


### ยืนยันได้ 4 ใน 5 คู่ — คู่ที่ 5 ต้องตรวจเพิ่ม

4 คู่จาก 5 คู่ใน P3 ปรากฏในผลลัพธ์ข้างบนตรงตามที่คาด (`รถยนต์นั่งส่วนบุคคล` ↔ `.../รถยนต์นั่งสาธารณะ`,
`รถบรรทุกไม่เกิน10ล้อ` ↔ `...มากกว่า6ล้อไม่เกิน10ล้อ`, `รถบรรทุกมากกว่า10ล้อ` ↔ `...(รถพ่วง)`,
`รถอื่นๆ` ↔ `อื่นๆ`) — ไม่มีแถวไหนเลยที่ทั้งคู่มีค่าพร้อมกัน ยืนยันว่า **coalesce ตรง ๆ ได้เลย**

แต่คู่ที่ 5 (`บริเวณที่เกิดเหตุ` ↔ `บริเวณที่เกิดเหตุอื่น`) **ไม่ปรากฏในตาราง** เพราะทั้งสองคอลัมน์
**มีค่าพร้อมกันถึง 7,606 แถว (4.7%)** — สมมติฐานแรก ("ชื่อเก่า/ชื่อใหม่เหมือน 4 คู่ข้างต้น") ยังไม่ถูก
พิสูจน์ว่าถูกหรือผิด — ต้องตรวจเพิ่มโดยตรง 2 สมมติฐาน: (1) เป็นชื่อเก่าที่ถูกแทนที่จริงไหม (2) คอลัมน์ที่สอง
เป็นช่องรายละเอียดที่ใช้เมื่อคอลัมน์แรกมีค่า "อื่นๆ" หรือไม่

### 4.1 ทดสอบสมมติฐานที่ 1 — `บริเวณที่เกิดเหตุ` ถูกแทนที่จริงไหม?

ถ้าเป็นคอลัมน์ชื่อเก่าที่ถูกแทนที่ (เหมือน 4 คู่แรก) ควรเห็น `บริเวณที่เกิดเหตุ` มีค่าเฉพาะปีเก่า ๆ
และหายไปในปีหลัง ๆ — ตรวจอัตราการมีค่าต่อปี

In [12]:
primary_col = "บริเวณที่เกิดเหตุ"
secondary_col = "บริเวณที่เกิดเหตุอื่น"

primary_by_year = population_rate_by_group(accidents_raw, "ปีที่เกิดเหตุ", primary_col)
secondary_by_year = population_rate_by_group(accidents_raw, "ปีที่เกิดเหตุ", secondary_col)

primary_by_year.join(secondary_by_year, on="ปีที่เกิดเหตุ", suffix="_secondary").select(
    "ปีที่เกิดเหตุ",
    "total_rows",
    pl.col("populated_rows").alias("primary_populated"),
    pl.col("populated_rows_secondary").alias("secondary_populated"),
)

ปีที่เกิดเหตุ,total_rows,primary_populated,secondary_populated
str,u32,u32,u32
"""2019""",19194,17659,0
"""2020""",21239,19530,0
"""2021""",20062,18074,0
"""2022""",20566,18281,0
"""2023""",23841,23515,0
"""2024""",23952,21660,0
"""2025""",23715,21265,0
"""2026""",8738,7882,8462


### 4.2 ทดสอบสมมติฐานที่ 2 — คอลัมน์ที่สองเป็นรายละเอียดของ "อื่นๆ" ไหม?

ในบรรดา 7,606 แถวที่ทั้งสองคอลัมน์มีค่า — ถ้าคอลัมน์ที่สองเป็นช่องรายละเอียดที่ใช้เฉพาะตอน
`บริเวณที่เกิดเหตุ = "อื่นๆ"` ควรเห็นค่าคอลัมน์แรกเป็น `"อื่นๆ"` เกือบทั้งหมด

In [13]:
hypothesis_check = check_value_when_both_populated(accidents_raw, primary_col, secondary_col, "อื่นๆ")

pl.DataFrame(
    [
        {
            "rows_both_populated": hypothesis_check.rows_both_populated,
            "primary_equals_อื่นๆ": hypothesis_check.matching_value_count,
            "pct": round(hypothesis_check.matching_pct, 1),
        }
    ]
)

rows_both_populated,primary_equals_อื่นๆ,pct
i64,i64,f64
7606,0,0.0


### สรุป P3 คู่ที่ 5 — ทั้งสองสมมติฐานผิด ไม่ใช่ twin column และไม่ใช่ช่อง "รายละเอียดของอื่นๆ"

**สมมติฐานที่ 1 (ชื่อเก่าถูกแทนที่) — ผิด:** `บริเวณที่เกิดเหตุ` มีค่าต่อเนื่องทุกปี 2019–2026
(91.7% โดยรวม, ในปี 2026 เองก็ยังมีค่าอยู่ 90.2%) ไม่เคยถูกเลิกใช้เลย

**สมมติฐานที่ 2 (ช่องรายละเอียดของ "อื่นๆ") — ผิดเช่นกัน:** ใน 7,606 แถวที่ทั้งคู่มีค่า
**0.0%** ที่ `บริเวณที่เกิดเหตุ = "อื่นๆ"` — คอลัมน์แรกมีค่าปกติทั่วไป (`ทางตรง+ไม่มีความลาดชัน`,
`ทางโค้งกว้าง+ไม่มีความลาดชัน` ฯลฯ) ในทุกแถวที่ตรวจ

**สิ่งที่ข้อมูลบอกจริง ๆ:** `บริเวณที่เกิดเหตุอื่น` มีค่า**เฉพาะใน `accident2026.csv`** เท่านั้น
(8,462 จาก 8,738 แถวของปีนั้น ไม่มีแม้แต่แถวเดียวในปีอื่น) และปรากฏ**คู่ขนาน**กับคอลัมน์แรก ไม่ใช่แทนที่
— ตัวอย่างค่าคู่กัน: primary `ทางตรง+ไม่มีความลาดชัน` ("ตรง ไม่มีลาดชัน") vs. secondary
`ทางตรง+ที่ราบ` ("ตรง พื้นราบ") — อธิบายแนวคิดเดียวกันด้วย**อนุกรมหมวดหมู่คนละชุด** อ่านได้ว่าระบบต้นทาง
เริ่มเก็บหมวดหมู่ใหม่ที่ละเอียดกว่าเดิมตั้งแต่ไฟล์ล่าสุด (2026) โดยเก็บคู่ขนานไปกับหมวดหมู่เดิม
— **ไม่ใช่การ rename และไม่ใช่ช่อง "อื่นๆ"**

**ยังไม่สรุปวิธีจัดการ — บันทึกไว้ใน CLAUDE.md ว่าต้องสืบสวนต่อใน Phase 2** (อาจต้องถามหน่วยงานต้นทางว่า
เปลี่ยนอะไรในปี 2026) ไม่ coalesce และไม่ยัดเข้าช่อง "รายละเอียดอื่นๆ" เพราะข้อมูลไม่สนับสนุนทั้งสองทาง

### 4.3 74% "conflict" ใน `บริเวณที่เกิดเหตุ` (จาก P5) น่าจะเป็นเรื่องเดียวกับหมวดหมู่คู่ขนานข้างบน

ที่ P5 พบว่า `บริเวณที่เกิดเหตุ` ขัดแย้งกันใน 74% ของคู่ซ้ำ (1,216 จาก 1,641 กลุ่ม) — คำถามคือ: ค่าที่
"ขัดแย้ง" กันนั้นเป็นตำแหน่งที่ต่างกันจริง หรือเป็นคำพ้องความหมายจากหมวดหมู่คนละชุด (เหมือนที่เจอใน
คอลัมน์คู่ขนานปี 2026)? สุ่มดู 20 คู่ตรง ๆ ก่อน

In [14]:
location_conflicts = find_conflicting_groups(accidents_raw, ["ACC_CODE"], "บริเวณที่เกิดเหตุ")
sample_20 = sample_deterministic(location_conflicts, 20, seed=settings.random_seed)

pl.DataFrame(
    [{"acc_code": key[0], "value_a": values[0], "value_b": values[1]} for key, values in sample_20]
)

acc_code,value_a,value_b
str,str,str
"""7605647""","""ทางตรง+ไม่มีความลาดชัน""","""ทางตรง+ไม่มีความลาดชัน/ที่ราบ"""
"""7540646""","""ทางสามแยก (T)""","""ทางสามแยก(T)"""
"""7668939""","""ทางตรง+ไม่มีความลาดชัน""","""ทางตรง+ไม่มีความลาดชัน/ที่ราบ"""
"""7736361""","""ทางตรง+ไม่มีความลาดชัน""","""ทางตรง+ไม่มีความลาดชัน/ที่ราบ"""
"""7630234""","""ทางตรง+ไม่มีความลาดชัน""","""ทางตรง+ไม่มีความลาดชัน/ที่ราบ"""
…,…,…
"""7630221""","""ทางตรง+ไม่มีความลาดชัน""","""ทางตรง+ไม่มีความลาดชัน/ที่ราบ"""
"""9032750""","""ทางโค้งกว้าง+ไม่มีความลาดชัน""","""ทางโค้งปกติ+ที่ราบ"""
"""7542751""","""ทางตรง+ไม่มีความลาดชัน""","""ทางตรง+ไม่มีความลาดชัน/ที่ราบ"""


ทั้ง 20 คู่ที่สุ่มมาเป็นรูปแบบเดิมซ้ำ ๆ: `"ไม่มีความลาดชัน"` (ไม่มีลาดชัน) กับ `"ที่ราบ"` (พื้นราบ) หรือ
`"X"` กับ `"X/ที่ราบ"` — คำพ้องความหมายหรือส่วนขยายรายละเอียด ไม่ใช่ตำแหน่งที่ต่างกันจริงสักคู่เดียว
ในตัวอย่างที่สุ่มมา ตรวจให้ครบทั้ง 1,216 คู่ด้วยกฎ normalize ง่าย ๆ (รวมคำพ้อง, ตัดส่วนขยายหลัง `/`,
ลบช่องว่าง) ด้านล่าง

In [15]:
resolved, still_conflicting = count_resolved_by_normalization(location_conflicts)

pl.DataFrame(
    [
        {
            "total_conflicts": len(location_conflicts),
            "resolved_as_synonym": resolved,
            "resolved_pct": round(resolved / len(location_conflicts) * 100, 1),
            "genuinely_different": still_conflicting,
            "genuinely_different_pct": round(still_conflicting / len(location_conflicts) * 100, 1),
        }
    ]
)

total_conflicts,resolved_as_synonym,resolved_pct,genuinely_different,genuinely_different_pct
i64,i64,f64,i64,f64
1216,1035,85.1,181,14.9


### สรุป — 74% "conflict" ส่วนใหญ่ไม่ใช่ conflict จริง เป็นเรื่องคำศัพท์

**85.1%** ของคู่ที่ขัดแย้งกัน (1,035/1,216) กลายเป็นค่าเดียวกันทันทีเมื่อ normalize คำพ้องความหมาย/
รูปแบบการเขียน — heuristic ที่ใช้นี้เป็นแบบง่ายมาก (rule เดียว ไม่ได้ปรับแต่งเจาะจงเป็นพิเศษ) ตัวเลขจริง
หลัง harmonize taxonomy อย่างละเอียดกว่านี้น่าจะสูงกว่านี้อีก — จาก 181 คู่ที่เหลือ ส่วนใหญ่ก็ยังเป็น
รูปแบบเดิมซ้ำ ๆ (เช่น `"ที่ลาดชัน"` vs `"โค้งดิ่ง/ที่ลาดชัน"`, `"ทางโค้งกว้าง"` vs `"ทางโค้งปกติ"`)
ไม่ใช่ 181 กรณีที่ต่างกันจริง ๆ

**ข้อสรุป (บันทึกไว้ใน CLAUDE.md แล้ว):** "conflict rate 74%" ที่รายงานใน P5 **ไม่ใช่สัญญาณของข้อมูล
ขัดแย้งกันจริง** — ส่วนใหญ่คือปัญหาคำศัพท์/หมวดหมู่ไม่ตรงกันระหว่างสองระบบการจัดหมวดหมู่ (แบบเดียวกับที่
เจอในคอลัมน์คู่ขนานของปี 2026) **ลำดับการแก้ต้องเป็น: harmonize taxonomy ก่อน แล้วค่อย deduplicate**
ไม่ใช่คิด tie-break rule ไปเลยตอนที่ยังไม่ได้รวมคำศัพท์ให้ตรงกัน — ลำดับสลับกันจะทำให้ทิ้งข้อมูลที่จริง ๆ
ตรงกันอยู่แล้วไปโดยไม่จำเป็น

## 5. รูปแบบวันที่/เวลาที่ปะปนกัน — ยืนยัน P1, P2

`classify_date_string` แยกค่าดิบในคอลัมน์ `วันที่เกิดเหตุ` เป็น 4 ประเภทต่อ `source_file` (ไม่พยายามแปลงเป็นวันที่จริง
— Phase 2 เท่านั้นที่ทำหน้าที่นั้น): `slash_unambiguous` (มีตัวเลขด้านใดด้านหนึ่ง > 12 จึงรู้ทันทีว่า DMY หรือ MDY),
`slash_ambiguous` (ทั้งสองด้าน ≤ 12 ต้องดูทั้งไฟล์ถึงจะสรุปได้), และ `excel_serial`

In [16]:
DATE_CATEGORIES = ["slash_unambiguous", "slash_ambiguous", "excel_serial", "empty", "other"]

date_format_rows = []
for source in sorted(accidents_raw["source_file"].unique().to_list()):
    sub = accidents_raw.filter(pl.col("source_file") == source)
    date_values = [v for v in sub["วันที่เกิดเหตุ"].to_list() if v is not None]
    counts = profile_date_column_formats(date_values)
    row = {"source_file": source} | {cat: counts.get(cat, 0) for cat in DATE_CATEGORIES}
    date_format_rows.append(row)

pl.DataFrame(date_format_rows)

source_file,slash_unambiguous,slash_ambiguous,excel_serial,empty,other
str,i64,i64,i64,i64,i64
"""accident2019.csv""",11846,7348,0,0,0
"""accident2020.csv""",12856,8382,1,0,0
"""accident2021.csv""",11751,8311,0,0,0
"""accident2022.csv""",12431,8135,0,0,0
"""accident2023.csv""",14578,9263,0,0,0
"""accident2024.csv""",0,0,23952,0,0
"""accident2025.csv""",0,0,23715,0,0
"""accident2026.csv""",4760,3978,0,0,0


### ตรงกับ CLAUDE.md P1 ทุกตัวเลข — พบ 1 แถวที่ผิดปกติเพิ่มเติม

ตัวเลข unambiguous/ambiguous ของทั้ง 8 `source_file` **ตรงกับตารางใน CLAUDE.md P1 แบบเป๊ะทุกตัว**
(เช่น accident2019: 11,846/7,348 — accident2024: excel_serial 23,952 ทั้งหมด)

แต่พบสิ่งที่ CLAUDE.md ยังไม่เคยบันทึกไว้: **`accident2020.csv` มี 1 แถว (`ACC_CODE=3810244`) ที่ค่า
วันที่เป็น Excel serial (`44178` = 2020-12-13) ทั้งที่ทั้งไฟล์ควรเป็น MM/DD/YYYY ล้วน** — วันที่แปลงแล้ว
ห่างจากวันที่รายงาน (`12/15/2020`) แค่ 2 วัน จึงสมเหตุสมผลว่าเป็นวันที่จริง เพียงแต่พิมพ์/บันทึกผิดรูปแบบ
1 แถวจาก 161,307 ไม่กระทบภาพรวม แต่เป็นตัวอย่างที่ดีว่าทำไม CLAUDE.md ถึงกำหนด **3-layer safety net**
ไว้สำหรับ Phase 2 (เทียบปีที่แปลงได้กับคอลัมน์ปีเดิม, ตรวจ accident_date ≤ report_date, ดู histogram
วันที่ 1-12) — กฎ per-file แบบ blanket จะพลาดแถวนี้ไปเฉย ๆ ถ้าไม่มี safety net ระดับแถว

In [17]:
TIME_CATEGORIES = ["clock_string", "excel_fraction", "empty", "other"]

time_format_rows = []
for source in sorted(accidents_raw["source_file"].unique().to_list()):
    sub = accidents_raw.filter(pl.col("source_file") == source)
    time_values = [v for v in sub["เวลา"].to_list() if v is not None]
    counts = profile_time_column_formats(time_values)
    row = {"source_file": source} | {cat: counts.get(cat, 0) for cat in TIME_CATEGORIES}
    time_format_rows.append(row)

pl.DataFrame(time_format_rows)

source_file,clock_string,excel_fraction,empty,other
str,i64,i64,i64,i64
"""accident2019.csv""",19194,0,0,0
"""accident2020.csv""",21238,1,0,0
"""accident2021.csv""",20062,0,0,0
"""accident2022.csv""",20566,0,0,0
"""accident2023.csv""",23841,0,0,0
"""accident2024.csv""",0,23952,0,0
"""accident2025.csv""",23715,0,0,0
"""accident2026.csv""",8738,0,0,0


### ยืนยัน P2 ครบ

`clock_string` (เช่น `21:10`) และ `excel_fraction` (เช่น `0.375` = 09:00) ปะปนกันตรงตามที่ CLAUDE.md
บันทึกไว้ และค่า `0:00` (ที่อาจเป็น "ไม่ทราบเวลา" มากกว่าเที่ยงคืนจริง) **ปรากฏ 219 ครั้ง — ตรงกับตัวเลขที่
บันทึกไว้เป๊ะ** ✅ (การตัดสินใจว่าจะ treat `0:00` เป็นเที่ยงคืนจริงหรือ "ไม่ทราบเวลา" ต้องเทียบกับ
คอลัมน์เวลาที่รายงาน — เป็นงานของ Phase 2)

## 6. จำนวนหมวดหมู่ (category cardinality) — ยืนยัน P4

ค่า `distinct` ที่คำนวณไว้แล้วใน section 2 ตรงกับ CLAUDE.md P4 ทุกคอลัมน์:
`มูลเหตุสันนิษฐาน` 120, `ลักษณะการเกิดเหตุ` 152, `บริเวณที่เกิดเหตุ` 89 — และตัวอย่างที่ยกไว้
(`ขับรถเร็วเกินอัตรากำหนด` 99,967 + `ขับรถเร็วเกินอัตราที่กำหนด` 14,636 = 114,603 แถว = 71.0% ของทั้งหมด)
**ตรงเป๊ะทุกตัวเลข** ✅

## 7. Profile ไฟล์ CSV ที่เหลือ

profile ไฟล์ฝนตกหนัก (rainfall), น้ำท่วม (flood), ฝนหนักมาก (R35mm), ทะเบียนรถ (stt_car_fuel),
และรถจดทะเบียนใหม่ (car_registered) — ดูจำนวนแถว/คอลัมน์ และ %null ของแต่ละคอลัมน์
(รายละเอียดครบทุกคอลัมน์อยู่ใน `reports/data_profiling_report.html`)

In [18]:
OTHER_CSV_SOURCES = [
    ("thailand_rainfall_2021_2026_combined.csv", None),
    ("thailand_flood_risk_areas.csv", ["GEOCODE", "Month"]),
    ("thailand_heavy_rain_R35mm.csv", ["ProvinceID", "Year"]),
    ("stt_car_fuel_at_25690228.csv", None),
    ("car_registered_new_2565_2569_combined.csv", None),
]

other_profiles = {
    filename: profile_csv_file(settings.raw_dir / filename, candidate_key_columns=key_cols)
    for filename, key_cols in OTHER_CSV_SOURCES
}

pl.DataFrame(
    [
        {
            "filename": filename,
            "rows": p.row_count,
            "columns": p.column_count,
            "full_row_dupes": p.duplicates.full_row_duplicate_count,
            "candidate_key_dupe_groups": p.duplicates.candidate_key_duplicate_group_count,
        }
        for filename, p in other_profiles.items()
    ]
)

filename,rows,columns,full_row_dupes,candidate_key_dupe_groups
str,i64,i64,i64,i64
"""thailand_rainfall_2021_2026_co…",30954,7,0,0
"""thailand_flood_risk_areas.csv""",27023,13,0,0
"""thailand_heavy_rain_R35mm.csv""",3465,4,0,0
"""stt_car_fuel_at_25690228.csv""",4643,5,0,0
"""car_registered_new_2565_2569_c…",969,6,0,0


## 8. โครงสร้างไฟล์ JSON ทั้ง 3 ไฟล์

`profile_json_structure` บอกว่า root เป็น list หรือ dict, มี key อะไรบ้าง, key ไหนเป็น array (ต้อง explode),
key ไหนเป็น object ซ้อน (nested), และความลึกสูงสุด — ใช้กับ `public_holidays_all.json`,
`thai_bank_holidays_all.json` (ทั้งคู่ root เป็น dict ที่มี `years` เป็น array ซ้อน array ของ holiday object)
และ `tat_tourist_attractions.json` (root เป็น list ของ record แบน ๆ)

In [19]:
JSON_FILENAMES = [
    "public_holidays_all.json",
    "thai_bank_holidays_all.json",
    "tat_tourist_attractions.json",
]

pl.DataFrame(
    [
        {
            "filename": filename,
            "root_type": s.root_type,
            "record_count": s.record_count,
            "max_depth": s.max_depth,
            "keys": ", ".join(s.keys),
            "array_valued_keys": ", ".join(s.array_valued_keys),
        }
        for filename, s in (
            (f, profile_json_structure(load_json_file(settings.raw_dir / f)))
            for f in JSON_FILENAMES
        )
    ]
)

filename,root_type,record_count,max_depth,keys,array_valued_keys
str,str,i64,i64,str,str
"""public_holidays_all.json""","""dict""",null,3,"""source, description, last_upda…","""available_years, years"""
"""thai_bank_holidays_all.json""","""dict""",null,3,"""source, description, last_upda…","""available_years, years"""
"""tat_tourist_attractions.json""","""list""",1731,1,"""ATT_ID, ATT_NAME_TH, ATT_NAME_…",""""""


## 9. สรุปผล

### ยืนยันตรงกับ CLAUDE.md §6 แบบเป๊ะทุกตัวเลข
- **P1** (รูปแบบวันที่ปะปน 3 แบบ) — ยืนยันครบทั้ง 8 `source_file`
- **P2** (เวลาแบบ `21:10` ปนกับ Excel fraction, `0:00` = 219 ครั้ง)
- **P4** (จำนวนหมวดหมู่ก่อนย่อ: 120 / 152 / 89 และตัวอย่างการรวม "เร่งความเร็ว" 71.0%)
- **P5 ตัวเลขพื้นฐาน** (แถวซ้ำเป๊ะ 82 แถว, ส่วนเกินจาก `ACC_CODE` ซ้ำ 1,702 แถว = 3,343 แถวใน 1,641 กลุ่ม
  ลบ 1,641, null 17.3%)
- **P3** (4 ใน 5 คู่คอลัมน์แฝดแยกกันสนิท 100% — `รถยนต์นั่งส่วนบุคคล`, `รถบรรทุกไม่เกิน10ล้อ`,
  `รถบรรทุกมากกว่า10ล้อ`, `รถอื่นๆ` แต่ละคู่ overlap=0 แถว, สัดส่วนมีค่าตรงกันทุกคู่ 82.7% เก่า / 5.4% ใหม่)
- ทุกไฟล์ CSV มี BOM ตรงตาม P7

### แก้ไข CLAUDE.md แล้วจากรอบการสืบสวนนี้ (สองรอบ — รอบแรกพบปัญหา รอบสองแก้ให้ครบ)

1. **P5 — ครบทุกแง่มุมแล้ว รวม safety net #4 ที่แก้ไขแล้ว:** หน่วยงานตรงกัน 100%, ยอดผู้เสียชีวิตตรงกัน
   100% (ไม่กำกวม) — ยอด reconciliation ใช้ **2 gate แยกกัน**: หลัง Extract 21,814 / หลัง Load (dedup)
   21,497 — ไม่ยุบเหลืออันเดียว วันที่ต่างกันใน 448 กลุ่มล้วนสลับวัน/เดือน อยู่ไฟล์เดียวกันเสมอ (ทั้งหมดใน
   `accident2023.csv`) — ยืนยันแล้วว่าไฟล์นี้เป็น MDY ล้วน (14,578 mdy_evidence / **0 dmy_evidence** ทั้งไฟล์
   ไม่ใช่แค่คะแนนเสียงข้างมาก) สมมติฐาน per-file **ใช้ได้ ไม่ต้อง redesign**
   **safety net #4 เดิมเขียนผิด** (จะ flag คู่ทั้ง 448 ว่า parser พังทุกครั้งโดยไม่มีทางเลี่ยง เพราะคู่ที่สลับ
   ตำแหน่งกันต้อง parse ต่างกันเสมอเมื่อไฟล์เป็น MDY ล้วน) — แก้เป็น **4a** (duplicate ที่ raw string เหมือน
   กันเป๊ะต้อง parse ตรงกัน — ตรวจแล้ว 1,193 กลุ่ม 0 mismatch) บวกรายงาน `date_ambiguous_pairs` แยกต่างหาก
   สำหรับ 448 คู่ที่สลับตำแหน่ง แก้ด้วย safety net #2 เป็นตัวชี้ขาด: **resolved 222 / both valid (ตัดสินไม่
   ได้) 196 / neither valid (flag) 30** — เสนอ `date_certainty` field สำหรับ Phase 2 (ยังไม่ implement)
2. **P5 conflict rate 74% ส่วนใหญ่ไม่ใช่ conflict จริง:** ตรวจ 20 คู่ตรง ๆ แล้ว **85.1%** ของคู่ที่ขัดแย้งกัน
   เป็นแค่คำพ้องความหมาย/รูปแบบเขียนต่างกัน (`"ไม่มีความลาดชัน"` ↔ `"ที่ราบ"` ฯลฯ) **ต้อง harmonize
   taxonomy ก่อน แล้วค่อย dedup** ไม่ใช่คิด tie-break rule ก่อน
3. **P3 คู่ที่ 5 ไม่ใช่ twin column เลย:** ทดสอบ 2 สมมติฐานตรง ๆ ทั้งคู่ผิด — `บริเวณที่เกิดเหตุ` ไม่เคย
   ถูกเลิกใช้ (มีค่าทุกปีถึง 2026) และไม่ใช่ช่องรายละเอียดของ "อื่นๆ" (0.0% ตรงเงื่อนไข) `บริเวณที่เกิดเหตุอื่น`
   เป็นหมวดหมู่คู่ขนานที่เริ่มเก็บเฉพาะปี 2026 ด้วยอนุกรมหมวดหมู่คนละชุด (ปรากฏการณ์เดียวกับข้อ 2 — สนับสนุน
   กันเอง) ยังไม่ coalesce, ต้องสืบสวนต่อใน Phase 2
4. **แถวผิดปกติ 1 แถวใน `accident2020.csv`:** วันที่เป็น Excel serial ทั้งที่ทั้งไฟล์ควรเป็นข้อความ MM/DD/YYYY

*(บันทึกไว้แล้วก่อนหน้านี้ใน Phase 1: P7 จำนวนพิกัดนอกประเทศไทยที่แท้จริงคือ 10 แถว ไม่ใช่ 15
เพราะ 5 แถวที่ lat ผิดซ้อนอยู่ในกลุ่ม lon ผิดด้วย — แก้ไขใน CLAUDE.md แล้วในรอบก่อนหน้า)*

### ขั้นต่อไป
Phase 1 ยังไม่แก้ไขข้อมูลใด ๆ — ผลจาก notebook นี้และ `reports/data_profiling_report.html`
เป็นหลักฐานยืนยันว่า 17 ปัญหาเป็นเรื่องจริง พร้อมส่งต่อให้ Phase 2 (ETL cleaning) ใช้ออกแบบกฎการทำความสะอาดต่อไป
— รวมถึงการตัดสินใจที่ยังค้างอยู่: ยืนยัน safety net ข้อ 4 ของ P1, ลำดับ harmonize-then-dedup ของ P5,
และวิธีจัดการ P3 คู่ที่ 5